# Notebook 04: Unquantized (bf16) DiffusionGemma split across two A100 40GB

**Scenario:** both GPUs hold part of the model (about 52 GB of bf16 weights split across 2 x 40 GB). Layers run in order, GPU 0 then GPU 1, so this is **slower than one GPU would be**: use it for correctness and quality, never for speed.

**What this notebook produces**

| # | Output | Hypothesis |
|---|---|---|
| 1 | Certified-sampler checks on real bf16 logits, computed live inside generation: decision mismatches (must be 0) and refinement rates per k | H4 |
| 2 | Top-k tail mass and entropy statistics per step and task | H2 |
| 3 | Unquantized accuracy baseline on a GSM8K subset: stock vs certified vs plain top-k | H2 |
| 4 | A small sample of raw logits saved for offline replay in Notebooks 01 and 03 | - |

Everything is measured **inside** the generation loop with a callback, so full-vocabulary logits never have to be stored for every step (about 134 MB per step otherwise).

## 1. Hardware check

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv
!nvidia-smi topo -m
!df -h . | tail -1

`topo -m` shows how the GPUs are linked: `NV#` means NVLink (fast), `PIX`/`PHB`/`SYS` mean PCIe (slower). It does not matter much here, since this split only passes activations between GPUs once per forward pass, but it matters for vLLM tensor parallelism in Notebook 06.

## 2. Install and shared code

In [ ]:
!pip install -q -U "transformers>=5.12" diffusers accelerate datasets pandas

In [ ]:
%%writefile dg_sampling.py
"""Shared sampler code for the DiffusionGemma top-k / certified sampling project.

Everything here is pure PyTorch and model-agnostic except TopKEntropyBoundScheduler,
which subclasses the diffusers EntropyBoundScheduler.
"""
import math
import torch
import torch.nn.functional as F

BUDGET, E_STOP = 0.1, 0.005


# ----------------------------------------------------------------------------- basic pieces
def chunked_logsumexp(x, tau, chunk=32_768):
    """log-sum-exp of x / tau over the last dim in fp32, one vocabulary chunk at a time."""
    out = None
    for s in range(0, x.shape[-1], chunk):
        part = torch.logsumexp(x[..., s:s + chunk].float() / tau, dim=-1)
        out = part if out is None else torch.logaddexp(out, part)
    return out


def exact_entropy(rows, tau):
    lp = torch.log_softmax(rows.float() / tau, dim=-1)
    return -(lp.exp() * lp).sum(-1)


def n_commit(e, b=BUDGET):
    s, _ = torch.sort(e)
    return int(((s.cumsum(0) - s) <= b).sum())


def commit_exact(e, b=BUDGET):
    order = torch.argsort(e, stable=True)
    keep = torch.zeros_like(e, dtype=torch.bool)
    keep[order[:n_commit(e, b)]] = True
    return keep


def topk_stats(logits, tau, k):
    """Exact top-k probabilities, tail mass m and a guaranteed entropy interval [lo, hi]."""
    Vv = logits.shape[-1]
    vals, idx = torch.topk(logits, k, dim=-1)
    lse = chunked_logsumexp(logits, tau)
    lp = vals.float() / tau - lse.unsqueeze(-1)
    p = lp.exp()
    h = -(p * lp).sum(-1)
    m = (1.0 - p.sum(-1)).clamp(0.0, 1.0)
    nmlm = -m * torch.log(m.clamp_min(1e-30))
    hi = h + nmlm + (m * math.log(Vv - k) if Vv > k else 0.0)
    lo = h - m * torch.log(torch.minimum(m, p[..., -1]).clamp_min(1e-30))
    return dict(idx=idx, p=p, m=m, lse=lse, h=h, lo=lo, hi=hi)


def topk_mix(st, E):
    """Self-conditioning mix over the top-k only: sum_k p_k * E[idx_k]."""
    B, Cc, k = st['p'].shape
    w = st['p'].reshape(-1, k).to(E.dtype).contiguous()
    try:
        out = F.embedding_bag(st['idx'].reshape(-1, k), E, per_sample_weights=w, mode='sum')
    except RuntimeError:   # some builds lack embedding_bag for this dtype: gather + weighted sum instead
        out = torch.einsum('nk,nkd->nd', w, E[st['idx'].reshape(-1, k)])
    return out.view(B, Cc, -1)


# ----------------------------------------------------------------------------- certified decisions
def certified_commit_row(lo, hi, refine, b=BUDGET, chunk=8):
    """Returns (commit mask identical to commit_exact(true entropies), refined slot count).
    Slots whose intervals are too loose to decide are refined with refine(pos) -> exact entropies."""
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if bool(exact.all()):
            return commit_exact(hi, b), refined
        n_lo, n_hi = n_commit(lo, b), n_commit(hi, b)
        if n_lo == n_hi:
            order = torch.argsort(hi, stable=True)
            S, R = order[:n_lo], order[n_lo:]
            if R.numel() == 0 or hi[S].max() < lo[R].min():
                keep = torch.zeros_like(exact)
                keep[S] = True
                return keep, refined
            amb = torch.cat([S[hi[S] >= lo[R].min()], R[lo[R] <= hi[S].max()]])
        else:
            amb = torch.argsort(lo, stable=True)[:n_lo + 1]
        amb = amb[~exact[amb]]
        if amb.numel() == 0:
            amb = (~exact).nonzero().squeeze(-1)
        if amb.numel() > chunk:
            amb = amb[torch.argsort(hi[amb] - lo[amb], descending=True)[:chunk]]
        e = refine(amb)
        lo[amb], hi[amb] = e, e
        exact[amb] = True
        refined += amb.numel()


def certified_stop_row(lo, hi, refine, e_stop=E_STOP, chunk=32):
    lo, hi = lo.clone(), hi.clone()
    exact = torch.zeros_like(lo, dtype=torch.bool)
    refined = 0
    while True:
        if hi.mean() <= e_stop:
            return True, refined
        if lo.mean() > e_stop:
            return False, refined
        cand = (~exact).nonzero().squeeze(-1)
        if cand.numel() == 0:
            return bool(hi.mean() <= e_stop), refined
        pick = cand[torch.argsort(hi[cand] - lo[cand], descending=True)[:chunk]]
        e = refine(pick)
        lo[pick], hi[pick] = e, e
        exact[pick] = True
        refined += pick.numel()


def certify_metrics(L, tau, ks=(16, 64, 256, 1024), b=BUDGET, e_stop=E_STOP):
    """For one step's logits L [B, C, V]: decision mismatches (must be 0) and refinement fractions per k."""
    e_true = exact_entropy(L, tau)
    rows = []
    for k in ks:
        st = topk_stats(L, tau, k)
        out = dict(k=k, commit_mismatch=0, stop_mismatch=0, refined_commit=0, refined_stop=0,
                   tail_mass_mean=st['m'].mean().item())
        for bi in range(L.shape[0]):
            refine = lambda pos, bi=bi: exact_entropy(L[bi, pos], tau)
            keep, r1 = certified_commit_row(st['lo'][bi], st['hi'][bi], refine, b)
            stop, r2 = certified_stop_row(st['lo'][bi], st['hi'][bi], refine, e_stop)
            out['commit_mismatch'] += int((keep != commit_exact(e_true[bi], b)).any())
            out['stop_mismatch'] += int(stop != bool(e_true[bi].mean() <= e_stop))
            out['refined_commit'] += r1
            out['refined_stop'] += r2
        n = L.shape[0] * L.shape[1]
        out['refined_commit_frac'] = out.pop('refined_commit') / n
        out['refined_stop_frac'] = out.pop('refined_stop') / n
        rows.append(out)
    return rows


# ----------------------------------------------------------------------------- diffusers integration
def calibrate_tau(sched, n_steps, device='cuda'):
    """Recover the stock scheduler's exact temperature at every timestep from pred_logits = logits / tau."""
    try:
        sched.set_timesteps(n_steps)
        ts = getattr(sched, 'timesteps', None)
        timesteps = [int(t) for t in ts] if ts is not None and len(ts) == n_steps else list(range(n_steps))
    except Exception:
        timesteps = list(range(n_steps))
    x = torch.randn(1, 8, 1024, device=device) * 3 + 1.0
    sample = torch.randint(1024, (1, 8), device=device)
    taus = [round((x / sched.step(x, t, sample).pred_logits.float()).median().item(), 6) for t in timesteps]
    return timesteps, taus


def make_scheduler_class():
    from diffusers import EntropyBoundScheduler
    try:
        from diffusers.schedulers.scheduling_entropy_bound import EntropyBoundSchedulerOutput
    except ImportError:
        from diffusers import EntropyBoundSchedulerOutput

    class TopKEntropyBoundScheduler(EntropyBoundScheduler):
        """Drop-in scheduler. mode: 'parent' | 'full' | 'topk' | 'certified'.
        Samples come from the top-k (no tail branch) in 'topk' and 'certified' modes."""

        def __init__(self, *args, mode='certified', k=64, tail='hi', pred_logits_mode='full',
                     tau_table=None, timesteps=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.mode, self.k, self.tail, self.pred_logits_mode = mode, k, tail, pred_logits_mode
            self.tau_by_t = dict(zip(timesteps, tau_table)) if (tau_table and timesteps) else None
            self.stats = []

        def _tau(self, timestep):
            t = int(timestep)
            if self.tau_by_t and t in self.tau_by_t:
                return self.tau_by_t[t]
            n = self.config.num_inference_steps
            return self.config.t_max - (self.config.t_max - self.config.t_min) * min(t, n - 1) / max(n - 1, 1)

        def step(self, model_output, timestep, sample, entropy_bound=None, generator=None, return_dict=True):
            if self.mode == 'parent':
                return super().step(model_output, timestep, sample, entropy_bound=entropy_bound,
                                    generator=generator, return_dict=return_dict)
            logits = model_output
            b = self.config.entropy_bound if entropy_bound is None else entropy_bound
            tau = self._tau(timestep)
            B, Cc, Vv = logits.shape
            g = generator if (generator is not None and generator.device == logits.device) else None
            refined = 0
            if self.mode == 'full':
                lp = torch.log_softmax(logits.float() / tau, -1)
                p = lp.exp()
                ent = -(p * lp).sum(-1)
                keep = torch.stack([commit_exact(ent[i], b) for i in range(B)])
                tok = torch.multinomial(p.view(-1, Vv), 1, generator=g).view(B, Cc)
                lse = torch.logsumexp(logits.float() / tau, -1)
            else:
                st = topk_stats(logits, tau, self.k)
                lse = st['lse']
                if self.mode == 'topk':
                    if self.tail == 'renorm':
                        q = st['p'] / st['p'].sum(-1, keepdim=True)
                        ent = -(q * torch.log(q.clamp_min(1e-30))).sum(-1)
                    else:
                        ent = {'lo': st['lo'], 'hi': st['hi'], 'mid': 0.5 * (st['lo'] + st['hi']), 'none': st['h']}[self.tail]
                    keep = torch.stack([commit_exact(ent[i], b) for i in range(B)])
                elif self.mode == 'certified':
                    rows = []
                    for i in range(B):
                        refine = lambda pos, i=i: exact_entropy(logits[i, pos], tau)
                        kp, r = certified_commit_row(st['lo'][i], st['hi'][i], refine, b)
                        rows.append(kp)
                        refined += r
                    keep = torch.stack(rows)
                else:
                    raise ValueError(self.mode)
                pick = torch.multinomial(st['p'].reshape(-1, self.k), 1, generator=g).view(B, Cc)
                tok = st['idx'].gather(-1, pick.unsqueeze(-1)).squeeze(-1)
            noise = torch.randint(Vv, (B, Cc), device=logits.device, generator=g)
            prev = torch.where(keep, tok, noise)
            sampled_probs = torch.exp(logits.gather(-1, tok.unsqueeze(-1)).squeeze(-1).float() / tau - lse)
            if self.pred_logits_mode == 'full':
                pred_logits = logits.float() / tau
            else:
                pred_logits = torch.full(logits.shape, float('-inf'), device=logits.device, dtype=torch.float32)
                vals, idx = torch.topk(logits, self.k, -1)
                pred_logits.scatter_(-1, idx, vals.float() / tau)
            self.stats.append(dict(t=int(timestep), committed=keep.sum(-1).float().mean().item(), refined=refined))
            out = EntropyBoundSchedulerOutput(prev_sample=prev, accepted_index=keep, sampled_tokens=tok,
                                              sampled_probs=sampled_probs, pred_logits=pred_logits)
            return out if return_dict else out.to_tuple()

    return EntropyBoundScheduler, TopKEntropyBoundScheduler


class StepRecorder:
    """diffusers callback: counts denoising steps and optionally runs a function on each step's logits."""
    def __init__(self, on_logits=None):
        self.on_logits = on_logits
        self.reset()

    def reset(self):
        self.steps = 0

    def __call__(self, pipe, step, timestep, kw):
        self.steps += 1
        if self.on_logits is not None and 'logits' in kw:
            self.on_logits(step, timestep, kw['logits'])
        return {'canvas': kw['canvas']}


# ----------------------------------------------------------------------------- small eval helpers
def last_number(s):
    import re
    nums = re.findall(r'-?\d[\d,]*\.?\d*', s)
    return nums[-1].replace(',', '').rstrip('.') if nums else None


def gsm8k_gold(ans):
    return ans.split('####')[-1].strip().replace(',', '')

In [ ]:
import time, os, json, gc
import torch, pandas as pd, transformers, diffusers
import dg_sampling as dgs
print('torch', torch.__version__, '| transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert torch.cuda.device_count() >= 2, 'this notebook expects two GPUs'
MODEL_ID = 'google/diffusiongemma-26B-A4B-it'
N_STEPS = 48
OUT_DIR = './runs/nb04'
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Load in bf16 across both GPUs

In [ ]:
from transformers import AutoProcessor, DiffusionGemmaForBlockDiffusion
from diffusers import DiffusionGemmaPipeline

t0 = time.time()
model = DiffusionGemmaForBlockDiffusion.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='auto',
    max_memory={0: '37GiB', 1: '37GiB'}, low_cpu_mem_usage=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f'loaded in {time.time() - t0:.0f}s')
placement = pd.Series(model.hf_device_map).value_counts()
print('modules per device:\n', placement)
for d in range(torch.cuda.device_count()):
    print(f'GPU {d}: {torch.cuda.memory_allocated(d) / 2**30:.1f} GB allocated')
assert not any(v in ('cpu', 'disk') for v in model.hf_device_map.values()), 'parts offloaded to CPU/disk: lower max_memory headroom or free GPU memory'

## 4. Pipeline, stock scheduler and exact temperature schedule

In [ ]:
StockSched, TopKSched = dgs.make_scheduler_class()
stock = StockSched(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS)
TIMESTEPS, TAU_TABLE = dgs.calibrate_tau(stock, N_STEPS, device='cuda:0')
print('TAU_TABLE first/last:', TAU_TABLE[:3], '...', TAU_TABLE[-3:])
json.dump(dict(timesteps=TIMESTEPS, tau_table=TAU_TABLE), open(f'{OUT_DIR}/tau_table.json', 'w'))
tau_of = dict(zip(TIMESTEPS, TAU_TABLE))

pipe = DiffusionGemmaPipeline(model=model, scheduler=stock, processor=processor)

def generate(pipe, prompt, rec, gen_length=256, seed=0):
    rec.reset()
    torch.cuda.synchronize(); t0 = time.time()
    out = pipe(prompt=prompt, gen_length=gen_length, num_inference_steps=N_STEPS,
               generator=torch.Generator('cuda').manual_seed(seed),
               callback_on_step_end=rec, callback_on_step_end_tensor_inputs=['canvas', 'logits'])
    torch.cuda.synchronize()
    return out, time.time() - t0

## 5. Certified-sampler checks on real bf16 logits (H4), live during generation

For each denoising step of each prompt, the callback computes, for k = 16, 64, 256, 1024: decision mismatches of the certified sampler vs. the full sampler (must be 0), refinement fractions, and mean tail mass. The stock scheduler drives generation, so the trajectories are the model's real ones.

The prompt set mixes tasks because confidence patterns differ between math, code and chat. Raw logits from the first prompt's first steps are saved for offline replay.

In [ ]:
PROMPTS = {
    'math': ['A store sells pencils at 3 for $1. How much do 27 pencils cost? Explain briefly.',
             'If a train travels 60 miles per hour for 2.5 hours, how far does it go?',
             'What is 17 * 23? Show the steps.'],
    'code': ['Write a Python function that returns the n-th Fibonacci number iteratively.',
             'Write a Python function that checks whether a string is a palindrome.',
             'Write a SQL query that returns the top 5 customers by total order value.'],
    'chat': ['Why is the sky blue? Answer in two sentences.',
             'Give three tips for writing a clear email.',
             'Explain what a hash table is to a beginner.'],
}
KS = (16, 64, 256, 1024)
SAVE_RAW = 6           # raw logits steps saved from the first prompt, for Notebooks 01 and 03
rows, saved = [], 0
ctx = {}

def on_logits(step, timestep, L):
    global saved
    tau = tau_of.get(int(timestep), TAU_TABLE[min(step, len(TAU_TABLE) - 1)])
    with torch.no_grad():
        for r in dgs.certify_metrics(L, tau, ks=KS):
            rows.append(dict(task=ctx['task'], prompt=ctx['i'], step=step, tau=tau, **r))
    if ctx['task'] == 'math' and ctx['i'] == 0 and saved < SAVE_RAW:
        torch.save(L.detach().to('cpu', torch.float16), f'{OUT_DIR}/step_{saved:02d}.pt'); saved += 1

rec = dgs.StepRecorder(on_logits=on_logits)
for task, prompts in PROMPTS.items():
    for i, p in enumerate(prompts):
        ctx.update(task=task, i=i)
        out, dt = generate(pipe, p, rec, gen_length=256, seed=i)
        print(f'[{task} {i}] {rec.steps} steps, {dt:.1f}s | {out.texts[0][:90]!r}')

cert = pd.DataFrame(rows)
cert.to_csv(f'{OUT_DIR}/certified_real_bf16.csv', index=False)
print('\nTOTAL decision mismatches:', int(cert.commit_mismatch.sum()), 'commit,', int(cert.stop_mismatch.sum()), 'stop  (must be 0)')
display(cert.groupby(['task', 'k'])[['refined_commit_frac', 'refined_stop_frac', 'tail_mass_mean']].mean().round(4))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, g in cert.groupby('k'):
    s = g.groupby('step').refined_commit_frac.mean()
    ax[0].plot(s.index, s.values, marker='.', label=f'k={k}')
    t = g.groupby('step').tail_mass_mean.mean()
    ax[1].plot(t.index, t.values, marker='.', label=f'k={k}')
ax[0].set(xlabel='denoising step', ylabel='fraction of slots refined', title='Refinement rate by step (real bf16 logits)')
ax[1].set(xlabel='denoising step', ylabel='mean tail mass', yscale='log', title='Probability outside the top-k')
for a in ax: a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/refinement_by_step.png', dpi=150); plt.show()

**Reading it:** mismatches must be exactly zero; anything else is a bug. A refinement rate of a few percent or less at k = 64 to 256 means the certified sampler is both exact in its decisions and cheap. Early steps (high entropy) are expected to need more refinement than late ones.

## 6. Unquantized accuracy baseline (GSM8K subset)

Stock vs. certified (k = 64) vs. plain renormalized top-k (k = 64), same prompts and seeds. `N_GSM` is small by default because bf16 split across two GPUs is slow; raise it for the final report.

In [ ]:
from datasets import load_dataset
N_GSM = 30
gsm = load_dataset('openai/gsm8k', 'main', split=f'test[:{N_GSM}]')
variants = {
    'stock':          stock,
    'certified k64':  TopKSched(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS,
                                mode='certified', k=64, tau_table=TAU_TABLE, timesteps=TIMESTEPS),
    'topk64 renorm':  TopKSched(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS,
                                mode='topk', k=64, tail='renorm', tau_table=TAU_TABLE, timesteps=TIMESTEPS),
}
plain = dgs.StepRecorder()
res = []
for name, sch in variants.items():
    pipe.scheduler = sch
    for j, ex in enumerate(gsm):
        prompt = ex['question'] + '\nSolve step by step, then give the final answer as a number.'
        out, dt = generate(pipe, prompt, plain, gen_length=512, seed=j)
        res.append(dict(variant=name, q=j, correct=dgs.last_number(out.texts[0]) == dgs.gsm8k_gold(ex['answer']),
                        steps=plain.steps, seconds=dt))
    print(name, 'done')
pipe.scheduler = stock
acc = pd.DataFrame(res)
acc.to_csv(f'{OUT_DIR}/gsm8k_bf16.csv', index=False)
display(acc.groupby('variant')[['correct', 'steps', 'seconds']].mean().round(3))

Seconds here are **not** a speed result (split model, scheduler-only change). What matters: `certified k64` should match `stock` in accuracy and steps up to sampling noise; `topk64 renorm` may take fewer steps and lose accuracy, which is the failure mode the certified sampler avoids.

## 7. Files written

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f, f'{os.path.getsize(os.path.join(OUT_DIR, f)) / 2**20:.1f} MB')